In [ ]:
"""
Fine-tune Qwen3-0.6B on the teacher-labeled tickets and compare it against the base model.

LoRA SFT, then a batched eval that prints priority and category accuracy for base vs tuned.
"""

import numpy as np
import pandas as pd


import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub

In [ ]:
import sys
# Unsloth patches transformers globally and broke generation here, so this notebook
# stays on the stock stack and guards against a stray import.
assert "unsloth" not in sys.modules, "Do not run any Unsloth cell in this notebook."
import torch, json, re, glob, os, random, gc
import pandas as pd
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
!pip install -q "transformers==4.55.2" "trl==0.21.0" "peft==0.17.0" "accelerate==1.10.0" "datasets>=2.20"

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from datasets import load_dataset

BASE  = "Qwen/Qwen3-0.6B"
TRAIN = glob.glob("/kaggle/input/**/train_chat.jsonl", recursive=True)[0]
print("train file:", TRAIN)

tok = AutoTokenizer.from_pretrained(BASE)
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(BASE)
# The KV cache is incompatible with gradient checkpointing, so turn it off for training.
model.config.use_cache = False

def to_text(ex):
    try:    t = tok.apply_chat_template(ex["messages"], tokenize=False, enable_thinking=False)
    # Older transformers builds do not take enable_thinking, so fall back without it.
    except TypeError: t = tok.apply_chat_template(ex["messages"], tokenize=False)
    return {"text": t}

train_ds = load_dataset("json", data_files=TRAIN, split="train").map(to_text, remove_columns=["messages"])
print("training examples:", len(train_ds))
assert len(train_ds) > 2000, "Expected ~2500 real rows, wrong file."

lora = LoraConfig(r=16, lora_alpha=16, lora_dropout=0.0, bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    task_type="CAUSAL_LM")
model = get_peft_model(model, lora)
model.enable_input_require_grads()
model.print_trainable_parameters()

In [ ]:
import torch
from transformers import Trainer, TrainingArguments

def _tok(ex):
    return tok(ex["text"], truncation=True, max_length=2048, add_special_tokens=False)
tok_ds = train_ds.map(_tok, remove_columns=train_ds.column_names)

# Pad positions get label -100 so the loss skips them. We pad on the right for
# training, the eval pads left for generation.
def collate(batch):
    pad = tok.pad_token_id
    L = max(len(b["input_ids"]) for b in batch)
    ii, am, lb = [], [], []
    for b in batch:
        x = b["input_ids"]; k = L - len(x)
        ii.append(x + [pad]*k); am.append([1]*len(x) + [0]*k); lb.append(x + [-100]*k)
    return {"input_ids": torch.tensor(ii),
            "attention_mask": torch.tensor(am),
            "labels": torch.tensor(lb)}

args = TrainingArguments(
    output_dir="out",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    learning_rate=2e-4,
    warmup_steps=5,
    lr_scheduler_type="linear",
    logging_steps=10,
    weight_decay=0.01,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    seed=3407,
    report_to="none",
    save_strategy="no",
)
args._n_gpu = 1

model.config.use_cache = False
trainer = Trainer(model=model, args=args, train_dataset=tok_ds, data_collator=collate)
trainer.train()

try: model.gradient_checkpointing_disable()
except Exception: pass
model.config.use_cache = True
print("training done, run the next two cells to rebuild the test set and evaluate.")

In [ ]:
from sklearn.model_selection import train_test_split
TL = glob.glob("/kaggle/input/**/test_labeled.jsonl", recursive=True)[0]
df = pd.read_csv("hf://datasets/Tobi-Bueck/customer-support-tickets/dataset-tickets-multi-lang-4-20k.csv")
q2c = {"Technical Support":"Technical Support","IT Support":"Technical Support",
       "Service Outages and Maintenance":"Technical Support","Product Support":"Product Support",
       "Customer Service":"Customer Service","General Inquiry":"Customer Service",
       "Human Resources":"Customer Service","Billing and Payments":"Billing & Payments",
       "Sales and Pre-Sales":"Billing & Payments","Returns and Exchanges":"Returns & Exchanges"}
df["category"] = df["queue"].map(q2c)
data = df[["subject","body","category","priority","language"]].dropna(
    subset=["body","category","priority"]).reset_index(drop=True)
# Rebuild the exact same split as the labeling notebook, same seeds, so each test
# row keeps its original index and lines up with its gold category/priority label.
# Note: this augment() draws random.choice before randint (the labeling notebook drew
# randint first), so the injected account sentences are NOT byte-identical to what the
# teacher saw. That is harmless here — the eval scores only category and priority, and
# neither depends on the account sentence — but don't reuse this rebuild to score
# account_id extraction.
train_pool, test = train_test_split(data, test_size=1200, stratify=data["category"], random_state=42)
train_split, _ = train_test_split(train_pool, train_size=2500, stratify=train_pool["category"], random_state=42)
random.seed(42)
en=["For reference, my account ends in {id}.","Same account I always use, the one ending {id}.",
    "My account number is {id}.","Please pull up account {id}."]
de=["Zur Info, mein Konto endet auf {id}.","Dasselbe Konto wie immer, endet auf {id}.","Meine Kontonummer ist {id}."]
def augment(frame, p=0.7):
    frame=frame.copy().reset_index(drop=True); out=[]
    for _,r in frame.iterrows():
        b=str(r["body"]).strip()
        if random.random()<=p:
            tpl=de if str(r["language"]).lower().startswith("de") else en
            b=b+" "+random.choice(tpl).format(id=str(random.randint(1000,99999)))
        out.append(b)
    frame["body_aug"]=out; return frame
train_split=augment(train_split); test=augment(test)
gold={}
for l in open(TL):
    r=json.loads(l)
    if r["label"]: gold[r["idx"]]=r["label"]
SYSTEM_SHORT=("Classify the support email. Respond with ONLY a JSON object with keys "
 '"category", "priority", and "account_id".\n'
 "category: one of Technical Support, Product Support, Customer Service, Billing & Payments, Returns & Exchanges.\n"
 "priority: one of high, medium, low.\n"
 'account_id: the account number stated in the email as a string, or null if none.')
rows=[]
for idx,r in test.iterrows():
    lab=gold.get(idx)
    if not lab or lab.get("category") is None or lab.get("priority") is None: continue
    rows.append({"user":f"Subject: {r['subject']}\n\nEmail: {r['body_aug']}",
                 "gold_cat":lab["category"],"gold_pri":lab["priority"]})
print("test rows ready:", len(rows))

In [ ]:
PRIS=["high","medium","low"]
def extract_json(t):
    for m in re.finditer(r"\{[^{}]*\}", t):
        try:
            j=json.loads(m.group(0))
            if "priority" in j or "category" in j: return j
        except Exception: pass
    return {}
def build_prompt(user):
    msgs=[{"role":"system","content":SYSTEM_SHORT},{"role":"user","content":user}]
    try:    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError: return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
# Left-pad for batched generation so each prompt ends flush against its new tokens.
def run_eval(m, bs=32):
    m.config.use_cache=True; m.eval(); tok.padding_side="left"; preds=[]
    # Qwen3 ships sampling defaults (temperature, top_p, top_k) but we decode greedily,
    # so clear them or transformers warns they are ignored on every batch.
    m.generation_config.do_sample = False
    m.generation_config.temperature = m.generation_config.top_p = m.generation_config.top_k = None
    for s in range(0,len(rows),bs):
        batch=rows[s:s+bs]
        enc=tok([build_prompt(r["user"]) for r in batch], return_tensors="pt",
                padding=True, truncation=True, max_length=2048).to("cuda")
        out=m.generate(**enc, max_new_tokens=80, do_sample=False, pad_token_id=tok.pad_token_id)
        for t in tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True):
            j=extract_json(t); p=j.get("priority")
            preds.append({"cat":j.get("category"),"pri":p.lower() if isinstance(p,str) else None})
        if s % (bs*5)==0: print(f"  {s+len(batch)}/{len(rows)}")
    return preds

base_model = AutoModelForCausalLM.from_pretrained(BASE).to("cuda")
print("BASE…");  base = run_eval(base_model)
del base_model; gc.collect(); torch.cuda.empty_cache()
print("FINE-TUNED…"); ft = run_eval(model)

def report(name, preds):
    gp=[r["gold_pri"] for r in rows]; pp=[p["pri"] for p in preds]
    gc_=[r["gold_cat"] for r in rows]; pc=[p["cat"] for p in preds]
    valid=sum(bool(p["cat"]) and bool(p["pri"]) for p in preds)
    print(f"\n===== {name} =====")
    print(f"valid JSON {valid}/{len(rows)} ({valid/len(rows):.1%}) | "
          f"priority {sum(a==b for a,b in zip(gp,pp))/len(rows):.1%} | "
          f"category {sum(a==b for a,b in zip(gc_,pc))/len(rows):.1%}")
    cm=pd.crosstab(pd.Series(gp,name="gold"),
                   pd.Series([p if p in PRIS else "none" for p in pp],name="pred"))
    print(cm.reindex(index=[x for x in PRIS if x in cm.index],
                     columns=[c for c in PRIS+["none"] if c in cm.columns], fill_value=0))
    return sum(a==b for a,b in zip(gp,pp))/len(rows)

print(f"\nbaseline (always 'high'): {sum(r['gold_pri']=='high' for r in rows)/len(rows):.1%}")
ba=report("BASE", base); fa=report("FINE-TUNED", ft)
print(f"\n>>> priority accuracy: {ba:.1%} -> {fa:.1%}  [+{(fa-ba)*100:.1f} pts]")

In [ ]:
# Merge the LoRA adapters into the base weights and save a 16-bit checkpoint.
# Run this after the eval cell. merge_and_unload consumes the PEFT model, so eval first.
import os
MERGED = "qwen3-triage-merged"

merged = model.merge_and_unload()               # fold LoRA into the base weights
merged = merged.half()                          # cast to fp16 for a 16-bit checkpoint
merged.save_pretrained(MERGED, safe_serialization=True)
tok.save_pretrained(MERGED)

print("merged 16-bit checkpoint saved ->", MERGED)
print("contents:", sorted(os.listdir(MERGED)))


In [ ]:
# Convert the merged checkpoint to GGUF q8_0 for LM Studio and llama.cpp.
import os, sys, glob, shutil, subprocess

if not os.path.isdir("llama.cpp"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp"], check=True)

# Install the gguf lib that matches this llama.cpp checkout plus the tokenizer deps,
# without upgrading the pinned transformers. That upgrade is what crashed the old convert.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "./llama.cpp/gguf-py", "sentencepiece", "protobuf"], check=True)

# Belt and suspenders: drop in the stock Qwen3 tokenizer so the converter cannot trip on
# tokenizer metadata. The fine-tune adds no new vocab, so it is byte-identical.
from huggingface_hub import snapshot_download
btok = snapshot_download("Qwen/Qwen3-0.6B",
    allow_patterns=["tokenizer*", "vocab.json", "merges.txt", "special_tokens_map.json", "added_tokens.json"])
for f in glob.glob(os.path.join(btok, "*")):
    if os.path.isfile(f): shutil.copy(f, MERGED)

os.makedirs("qwen3-triage-gguf", exist_ok=True)
OUT = "qwen3-triage-gguf/triage-0.6b-q8_0.gguf"
subprocess.run([sys.executable, "llama.cpp/convert_hf_to_gguf.py", MERGED,
                "--outfile", OUT, "--outtype", "q8_0"], check=True)

sz = os.path.getsize(OUT) / 1e6
print()
print(f"GGUF written: {OUT}  ({sz:.0f} MB)")
